# LAB 07 — Lakeflow Spark Declarative Pipelines

## Scenario

> *"Build a complete Medallion Pipeline using Lakeflow Spark Declarative Pipelines. First you'll create a pipeline via the Databricks UI (Workshop), then practice writing SQL declarations for Bronze, Silver, and Gold layers (Practice)."*



## Objectives

After completing this lab you will be able to:
- Create and configure a Lakeflow Spark Declarative Pipeline in the Databricks UI
- Upload SQL source files and set pipeline variables
- Write `STREAMING TABLE` declarations for Bronze
- Add data quality expectations (`ON VIOLATION DROP ROW`)
- Create `MATERIALIZED VIEW` declarations for Gold
- Verify pipeline results and query the Event Log



## Prerequisites

- Cluster running and attached to notebook
- SQL source files available in `materials/lakeflow/lakeflow_demo/transformations/` (`01_bronze/`, `02_silver/`, `03_gold/`)



## Section 1: Workshop — Building the Pipeline in UI

Follow the trainer's instructions step by step.

### Step 1: Upload SQL Files
- Source folder: `materials/lakeflow/lakeflow_demo/transformations/` with subfolders `01_bronze/`, `02_silver/`, `03_gold/`
- **Option A (UI upload):** create a `lakeflow_demo/transformations/` folder in your workspace user folder, recreate the three subfolders and upload the SQL files from the matching repo folders
- **Option B (Git folder):** clone the training repository as a Git folder — the source folder is `<repo>/materials/lakeflow/lakeflow_demo/transformations/`

### Step 2: Create Pipeline
1. Go to **Jobs & Pipelines → Create → ETL pipeline**
2. Set **Pipeline name**: `lakeflow_pipeline_<your_name>`
3. Set **Catalog**: `retailhub_<your_name>` (the `CATALOG` printed by the Setup cell)
4. Set **Target schema**: `<your_name>_lakeflow` (the `Pipeline target schema` printed by the Setup cell)
5. Add existing assets: **Pipeline root folder** = the `lakeflow_demo` folder; **Source code path** = its `transformations/` folder

### Step 3: Configure Variables
1. **Settings** → **Pipeline configuration**
2. **Add configuration**
3. Enter the keys (replace `<your catalog>` with your catalog):
   - `customer_path` → `/Volumes/<your catalog>/default/datasets/customers`
   - `order_path` → `/Volumes/<your catalog>/default/datasets/orders`
   - `product_path` → `/Volumes/<your catalog>/default/datasets/products/products.parquet/`
4. **Save** — the DAG appears

### Step 4: Run the Pipeline
1. Click **Start** and wait until the run shows **Completed**
2. Copy one new file from `datasets/demo/ingestion/orders/stream/` (e.g. `orders_stream_004.json`) into `datasets/orders/stream/` — via Catalog Explorer download/upload, or:
   ```python
   dbutils.fs.cp(
       "/Volumes/<your_catalog>/default/datasets/demo/ingestion/orders/stream/orders_stream_004.json",
       "/Volumes/<your_catalog>/default/datasets/orders/stream/orders_stream_004.json",
   )
   ```
3. Click **Start** again — only the new file is processed
4. Check the **Event Log** tab for processing metrics

### Step 5: Verify Results
- Query `fact_sales` with joins to `dim_customer`, `dim_product`, `dim_date`
- Check SCD Type 2 history in `silver_customers`



## Section 2: Practice — Lakeflow SQL Declarations

Open the lab notebook **`lab_07_lakeflow_pipeline.ipynb`** (in `notebooks/day2/lab/`) and complete the `# TODO` cells.

| Task | What to do | Key concept |
|------|-----------|-------------|
| **Task 1** | Write Bronze Declaration | `CREATE OR REFRESH STREAMING TABLE` + `read_files()` |
| **Task 2** | Write Silver with Expectations | `ON VIOLATION DROP ROW` for constraints |
| **Task 3** | Write Gold Declaration | `CREATE OR REFRESH MATERIALIZED VIEW` |
| **Task 4** | Classify ST vs MV | Fill `answer` dict, confirm via `information_schema.tables.table_type` |
| **Task 5** | Verify Pipeline Results | Row counts: bronze ≥ silver == fact |
| **Task 6** | Check Pipeline Event Log | `event_log(TABLE(...))` for data quality metrics |
| **Task 7** | Data-quality quarantine | dropped rows = bronze − silver; silver has 0 violations |



## Detailed Hints

### Task 1: Bronze — STREAMING TABLE
- Replace all three `____` placeholders — the check fails while any remain
- `CREATE OR REFRESH STREAMING TABLE bronze_orders`
- Source: `STREAM read_files('/Volumes/{CATALOG}/default/datasets/orders/stream/', format => 'json')`

### Task 2: Silver — Expectations
- Constraints use `EXPECT (condition) ON VIOLATION DROP ROW`
- First constraint: `order_id IS NOT NULL`
- Second constraint: `total_price > 0`
- Source: `STREAM(bronze_orders)`

### Task 3: Gold — MATERIALIZED VIEW
- `CREATE OR REFRESH MATERIALIZED VIEW gold_daily_revenue`
- Materialized Views are recalculated from scratch each run

### Task 4: ST vs MV
| Feature | Streaming Table | Materialized View |
|---------|----------------|-------------------|
| Processing mode | Incremental (append) | Full recompute |
| Best for | Raw/Bronze ingestion | Aggregations/Gold |

### Task 5: Verify Pipeline Results
- Count each layer with `spark.table(...)` + `.count()`; full names are `{CATALOG}.{user_schema}.<table>`
- Expect bronze ≥ 100k (batch backfill), silver ≤ bronze (DROP ROW), fact == silver

### Task 6: Event Log
- `SELECT * FROM event_log(TABLE(catalog.schema.table))`
- Filter by `event_type = 'flow_progress'` for data quality metrics
- The event log is pipeline-wide — keep only `silver_orders` rows (`exp.dataset` ends with `.silver_orders`) before grouping by constraint name; expect 5 constraints, with failures only on `valid_order_id` and `valid_customer`

### Task 7: Data-quality quarantine
- `dropped_rows` / `dropped_pct` reuse `bronze_cnt` and `silver_cnt` from Task 5 (percentage of bronze)
- Apply the provided `VIOLATION_PREDICATE` with `.filter(...)` on bronze and silver and `.count()` — bronze must equal `dropped_rows`, silver must be 0
- Expect ~6% dropped: rows with a null `order_id` or a null `customer_id` (the quantity / product / unit-price rules drop nothing — negative quantities are returns and pass by design)



## Summary

In this lab you:
- Created a complete Lakeflow Spark Declarative Pipeline via Databricks UI
- Wrote SQL declarations for Bronze (STREAMING TABLE), Silver (with expectations), and Gold (MATERIALIZED VIEW)
- Verified incremental processing with streaming sources
- Queried the Event Log for pipeline monitoring

> **Exam Tip:** `STREAMING TABLE` processes data incrementally (append-only). `MATERIALIZED VIEW` recalculates fully each run. Use `ON VIOLATION DROP ROW` to silently filter invalid rows. `ON VIOLATION FAIL UPDATE` stops the update. `EXPECT` without action only logs warnings.

> **What's next:** In LAB 08 you will create multi-task Jobs with dependencies and triggers for orchestrating pipelines.